# Earth Observation Data and Tools for Climate Resilience and Risk Monitoring
## with medium-resolution EO examples for Africa and Asia

This notebook is focused on:
- medium-resolution Earth Observation (EO) datasets such as **Sentinel-3**, **MODIS**, and a few complementary products;
- practical access patterns using **STAC API**, **Microsoft Planetary Computer (MPC)**, **Google Earth Engine (GEE)**, **Xee**, and **cubo**;
- simple climate resilience and risk-monitoring examples for **Africa** and **Asia**.

### Tutorial structure
1. Why medium-resolution EO for resilience and risk monitoring
2. Data access options: STAC, MPC, GEE, Xee, cubo
3. Example A (Africa): vegetation and drought context with MODIS NDVI
4. Example B (Asia): coastal and delta monitoring with Sentinel-3 OLCI
5. Integrated approach: combine EO with climate context
6. Wrap-up, limitations, and next steps

> This notebook is built as a **teaching notebook**. Some cells may require authentication or internet access and are meant for live demonstration rather than offline execution.

## Learning objectives

By the end of the session, participants should be able to:
- explain why medium-resolution EO is useful for continental and regional monitoring;
- discover datasets with a **STAC API**;
- read EO data from **Microsoft Planetary Computer**;
- access Earth Engine collections in Python with **GEE** and **Xee**;
- use **cubo** as a higher-level data-cube interface for both STAC and GEE;
- build basic resilience indicators such as:
  - NDVI time series,
  - simple anomalies,
  - rainfall + vegetation context,
  - coastal water/land color quicklooks.

## Why medium-resolution EO?

Medium-resolution EO is often a strong fit for climate resilience because it balances:
- **spatial detail** sufficient for regional landscapes, coastlines, croplands, and large water bodies;
- **temporal frequency** suitable for anomaly detection and seasonal monitoring;
- **manageable data volume** for teaching and prototyping.

### Typical use cases
- drought and vegetation stress,
- flood extent context,
- water quality and coastal monitoring,
- land surface conditions,
- seasonal food-security and livelihood monitoring.

## Datasets used in this tutorial

### Core datasets
- **MODIS MOD13Q1 / MYD13Q1**: 16-day vegetation indices at **250 m**
- **Sentinel-3 OLCI**: ocean and land color radiances at **300 m**
- optional complementary context:
  - **CHIRPS** rainfall,
  - **MODIS LST**,
  - **Sentinel-3 SLSTR** where relevant

### Study areas
- **Africa**: Lake Victoria basin / East Africa vegetation monitoring
- **Asia**: Ganges-Brahmaputra delta or Mekong delta coastal-water example

You can replace these AOIs with your own polygons later.

In [1]:

# !pip install -q cubo xarray dask[complete] pystac-client planetary-computer xee earthengine-api geemap matplotlib pandas numpy shapely
#
# Optional depending on your workflow:
# !pip install -q odc-stac stackstac rioxarray

## Imports

In [2]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from shapely.geometry import box, mapping

# STAC / MPC
from pystac_client import Client
import planetary_computer as pc

# GEE / Xee
import ee
import xee

# Cubo
import cubo

ModuleNotFoundError: No module named 'numpy'

## Authentication notes

### Earth Engine
For local Jupyter:
```python
ee.Authenticate()
ee.Initialize(project="YOUR_GEE_PROJECT")
```

For high-volume endpoint when appropriate:
```python
ee.Initialize(project="YOUR_GEE_PROJECT",
              opt_url="https://earthengine-highvolume.googleapis.com")
```

### Microsoft Planetary Computer
The STAC API is public for metadata search. Asset access often requires **signing** with `planetary_computer.sign_inplace(...)` or a modifier function in `pystac-client`.

In [ ]:

# Uncomment when running locally for the first time:
# ee.Authenticate()

# Example initialization:
# ee.Initialize(project="YOUR_GEE_PROJECT")

# Or high-volume endpoint:
# ee.Initialize(
#     project="YOUR_GEE_PROJECT",
#     opt_url="https://earthengine-highvolume.googleapis.com"
# )

## Helper variables: Africa and Asia AOIs

In [ ]:

# Africa example: Lake Victoria region (broad bbox)
africa_bbox = [32.0, -3.5, 35.8, 0.8]   # minx, miny, maxx, maxy

# Asia example: Ganges-Brahmaputra delta / coastal Bangladesh (broad bbox)
asia_bbox = [88.0, 21.0, 91.5, 24.0]

africa_geom = mapping(box(*africa_bbox))
asia_geom = mapping(box(*asia_bbox))

print("Africa bbox:", africa_bbox)
print("Asia bbox:", asia_bbox)

# Part 1 — Discover data with STAC API and MPC

This section shows the lowest-friction pattern:
1. open a STAC API,
2. search a collection by space and time,
3. inspect returned items,
4. optionally pass those results into higher-level tooling later.

In [ ]:

# Microsoft Planetary Computer STAC
stac_url = "https://planetarycomputer.microsoft.com/api/stac/v1"

catalog = Client.open(
    stac_url,
    modifier=pc.sign_inplace,  # automatically sign returned items/assets
)

print(catalog)

## Example 1A: Search MODIS NDVI for Africa

In [ ]:

search_modis_africa = catalog.search(
    collections=["modis-13Q1-061"],
    bbox=africa_bbox,
    datetime="2022-01-01/2023-12-31",
    limit=5,
)

items_modis_africa = list(search_modis_africa.items())
print(f"Returned {len(items_modis_africa)} items")
for item in items_modis_africa[:3]:
    print(item.id, item.datetime)

### What to point out during teaching
- STAC separates **collection metadata** from **items/assets**.
- A STAC search is often the most transferable pattern across cloud EO catalogs.
- MPC is a good demonstration platform because it exposes many datasets through a standard STAC API.

## Example 1B: Search Sentinel-3 for Asia

In [ ]:

# Depending on the exact teaching goal, you may choose one of several Sentinel-3 collections.
# Here we use a generic Sentinel-3 OLCI-related example available through MPC's STAC.

search_s3_asia = catalog.search(
    collections=["sentinel-3-olci-lfr-l2-netcdf"],
    bbox=asia_bbox,
    datetime="2023-01-01/2023-03-31",
    limit=5,
)

items_s3_asia = list(search_s3_asia.items())
print(f"Returned {len(items_s3_asia)} items")
for item in items_s3_asia[:3]:
    print(item.id, item.datetime)

## Optional: Inspect asset names from a returned STAC item

In [ ]:

if items_modis_africa:
    first_item = items_modis_africa[0]
    print("Item ID:", first_item.id)
    print("Assets:")
    for k in first_item.assets:
        print(" -", k)

# Part 2 — Use `cubo` as a simpler data-cube interface

`cubo` is handy for teaching because it gives a compact, xarray-friendly entry point and supports **STAC** and **GEE** workflows.

## Example 2A: `cubo` with STAC-like workflow (Africa)

This is a compact pattern for building an on-demand EO mini-cube around a point.
For live teaching, keep:
- the cube small,
- the time range short,
- the number of bands minimal.

In [ ]:

# A point near Lake Victoria
lat_africa = -1.0
lon_africa = 33.0

# This cell is illustrative and may require internet access.
# Collection names depend on the backend known by cubo.
# Start with a small cube for a live demo.

# da_africa_cubo = cubo.create(
#     lat=lat_africa,
#     lon=lon_africa,
#     collection="modis-13Q1-061",
#     bands=["250m_16_days_NDVI"],
#     start_date="2023-01-01",
#     end_date="2023-06-30",
#     edge_size=64,
#     resolution=250,
# )
# da_africa_cubo

### Teaching note
If a collection name above is not resolved in your environment, keep the concept cell and switch the live demo to:
- a collection known to work in your cubo installation, or
- the GEE-backed cubo example below.

This is one reason to keep **multiple access options** in the tutorial.

## Example 2B: `cubo` with Google Earth Engine (Asia, Sentinel-3 or MODIS)

When `gee=True`, `cubo` can pull data cubes from Earth Engine collections.

In [ ]:

# Initialize Earth Engine first if needed:
# ee.Initialize(project="YOUR_GEE_PROJECT",
#               opt_url="https://earthengine-highvolume.googleapis.com")

lat_asia = 22.5
lon_asia = 90.0

# Example: MODIS NDVI via GEE-backed cubo
# da_asia_cubo = cubo.create(
#     lat=lat_asia,
#     lon=lon_asia,
#     collection="MODIS/061/MOD13Q1",
#     bands=["NDVI", "EVI"],
#     start_date="2023-01-01",
#     end_date="2023-12-31",
#     edge_size=64,
#     resolution=250,
#     gee=True,
# )
# da_asia_cubo

### Optional Sentinel-3 variant in GEE-backed cubo

In [ ]:

# Sentinel-3 OLCI in GEE
# da_s3_cubo = cubo.create(
#     lat=22.5,
#     lon=90.0,
#     collection="COPERNICUS/S3/OLCI",
#     bands=["Oa08_radiance", "Oa06_radiance", "Oa04_radiance"],
#     start_date="2023-01-01",
#     end_date="2023-01-31",
#     edge_size=64,
#     resolution=300,
#     gee=True,
# )
# da_s3_cubo

# Part 3 — Earth Engine + Xee for xarray-native analysis

`xee` is excellent when you want Earth Engine collections to feel more like xarray datasets.
This is a strong teaching path for people already comfortable with Python arrays.

## Example 3A: Africa MODIS NDVI time series with Xee

In [ ]:

# Earth Engine geometry
africa_roi = ee.Geometry.Rectangle(africa_bbox)

# MODIS NDVI collection
modis_ic = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterBounds(africa_roi)
    .filterDate("2023-01-01", "2023-12-31")
    .select(["NDVI", "EVI"])
)

# Open with xee
# ds_modis_xee = xr.open_dataset(
#     modis_ic,
#     engine="ee",
#     geometry=africa_bbox,
#     scale=250,
# )
# ds_modis_xee

## Example 3B: Reduce to a simple regional mean time series

In [ ]:

# Example post-processing once ds_modis_xee is loaded:
#
# ndvi = ds_modis_xee["NDVI"] / 10000.0  # MODIS NDVI is typically scaled
# ndvi_mean = ndvi.mean(dim=("X", "Y")).to_series()
#
# ax = ndvi_mean.plot(figsize=(10, 4), marker="o")
# ax.set_title("Lake Victoria region — MODIS NDVI mean (2023)")
# ax.set_ylabel("NDVI")
# plt.show()

## Example 3C: Simple anomaly against a baseline

In [ ]:

# Illustrative anomaly workflow
#
# modis_baseline_ic = (
#     ee.ImageCollection("MODIS/061/MOD13Q1")
#     .filterBounds(africa_roi)
#     .filterDate("2015-01-01", "2022-12-31")
#     .select("NDVI")
# )
#
# ds_baseline = xr.open_dataset(
#     modis_baseline_ic,
#     engine="ee",
#     geometry=africa_bbox,
#     scale=250,
# )
#
# baseline_mean = (ds_baseline["NDVI"] / 10000.0).mean(dim="time")
# current_mean  = (ds_modis_xee["NDVI"] / 10000.0).mean(dim="time")
# anomaly = current_mean - baseline_mean
#
# anomaly.plot(figsize=(6, 5), cmap="RdYlGn")
# plt.title("NDVI anomaly: 2023 vs 2015-2022 baseline")
# plt.show()

# Part 4 — Asia example: Sentinel-3 OLCI for coastal / delta monitoring

Sentinel-3 OLCI is useful for:
- coastal water color context,
- sediment/turbidity proxies,
- large estuaries and deltas,
- broad land-water transitions.

For a beginner tutorial, use it as a **quicklook and exploratory cube** rather than a full biophysical retrieval.

In [ ]:

asia_roi = ee.Geometry.Rectangle(asia_bbox)

s3_ic = (
    ee.ImageCollection("COPERNICUS/S3/OLCI")
    .filterBounds(asia_roi)
    .filterDate("2023-01-01", "2023-01-31")
    .select(["Oa08_radiance", "Oa06_radiance", "Oa04_radiance"])
)

# ds_s3 = xr.open_dataset(
#     s3_ic,
#     engine="ee",
#     geometry=asia_bbox,
#     scale=300,
# )
# ds_s3

## Quicklook visualization idea

In [ ]:

# Example after ds_s3 is loaded:
#
# rgb = xr.concat(
#     [
#         ds_s3["Oa08_radiance"].isel(time=0),
#         ds_s3["Oa06_radiance"].isel(time=0),
#         ds_s3["Oa04_radiance"].isel(time=0),
#     ],
#     dim="band"
# )
#
# rgb = rgb / rgb.quantile(0.98)
# rgb = rgb.clip(0, 1)
#
# plt.figure(figsize=(6, 6))
# plt.imshow(np.transpose(rgb.values, (1, 2, 0)))
# plt.title("Sentinel-3 OLCI quicklook — coastal Bangladesh")
# plt.axis("off")
# plt.show()

# Part 5 — Integrated approach: combine EO with climate context

A resilience workflow is usually stronger when satellite observations are interpreted together with:
- rainfall,
- temperature,
- hydrology,
- exposure and vulnerability layers.

Below is a basic rainfall + vegetation example.

## Example 5A: CHIRPS rainfall + MODIS NDVI in Africa

In [ ]:

# CHIRPS daily rainfall in Earth Engine
chirps_ic = (
    ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
    .filterBounds(africa_roi)
    .filterDate("2023-01-01", "2023-12-31")
    .select("precipitation")
)

# ds_chirps = xr.open_dataset(
#     chirps_ic,
#     engine="ee",
#     geometry=africa_bbox,
#     scale=5566,  # approximate native grid
# )
# ds_chirps

In [ ]:

# Example combined plot once ds_modis_xee and ds_chirps are available:
#
# rainfall = ds_chirps["precipitation"].mean(dim=("X", "Y")).to_series()
# ndvi = (ds_modis_xee["NDVI"] / 10000.0).mean(dim=("X", "Y")).to_series()
#
# fig, ax1 = plt.subplots(figsize=(11, 4))
# rainfall.plot(ax=ax1, kind="bar", alpha=0.6)
# ax1.set_ylabel("Rainfall")
# ax1.set_title("Africa example — rainfall and vegetation context")
#
# ax2 = ax1.twinx()
# ndvi.plot(ax=ax2, color="darkgreen", marker="o")
# ax2.set_ylabel("NDVI")
# plt.show()

## Example 5B: Discussion prompts for the audience

Ask participants:
1. Which platform felt easiest for discovery?
2. Which path gave the most analysis-ready xarray object?
3. Where are the hidden complexities?
   - scaling factors,
   - QA bands,
   - authentication,
   - reprojection,
   - temporal aggregation,
   - cloud/noise filtering.

# Comparison of the four main access patterns

| Option | Best use | Strength | Watch out for |
|---|---|---|---|
| STAC API | discovery and interoperable search | standards-based, portable | actual asset handling may still require extra code |
| MPC | cloud-hosted STAC datasets | easy search and many public datasets | collection naming and asset structure vary |
| GEE | large-scale catalog + processing | huge catalog, server-side filtering | authentication and API concepts |
| Xee | Earth Engine to xarray | analysis-friendly for Python users | some workflows still need EE thinking |
| cubo | compact data-cube creation | simple mental model across STAC/GEE | backend support varies by collection |

# Recommended live-demo strategy

For a smooth 90-minute session, do this:

### Primary live path
1. **STAC + MPC**: show dataset discovery
2. **GEE + Xee**: load MODIS and compute a regional NDVI time series
3. **cubo + GEE**: show the same idea with less code
4. **Sentinel-3 quicklook**: coastal or delta example in Asia

### Backup path
If any service-specific cell is slow or authentication fails:
- switch to metadata search,
- show the code pattern,
- explain expected outputs using screenshots prepared beforehand.

# Suggested speaking notes

### Key message
Climate resilience monitoring is rarely about one sensor or one platform. The most practical skill is learning to move between:
- discovery,
- access,
- cube creation,
- aggregation,
- interpretation.

### Interpretation examples
- Lower NDVI together with rainfall deficits can indicate drought stress.
- Sentinel-3 coastal color patterns can help identify sediment plumes and broad coastal change context.
- Medium-resolution products are excellent for regional monitoring, but not for parcel-scale decisions.

# Exercises for participants

1. Replace the Africa bounding box with a Sahel AOI.
2. Replace the Asia AOI with the Mekong Delta.
3. Add a 2019–2024 annual NDVI summary.
4. Compute a monthly climatology.
5. Add MODIS LST or CHIRPS rainfall and compare seasonal patterns.
6. Try the same workflow through:
   - raw STAC search,
   - Xee,
   - cubo.

# Closing remarks

This notebook intentionally mixes several access patterns because that reflects real EO practice:
- no single interface solves every workflow,
- standards like STAC improve portability,
- Earth Engine accelerates computation,
- xarray-based tools make Python analysis cleaner,
- `cubo` can reduce boilerplate for teaching and prototyping.

For production work, add:
- QA masking,
- uncertainty checks,
- reprojection control,
- robust anomaly baselines,
- validation with local/contextual data.